In [ ]:
%%sql
SELECT DISTINCT
    a.appointment_status AS session_status_src_id,
    m.mapping AS session_status_src_name,
    'S1' AS session_status_src_sys_inst_id
FROM silver_sone_srappointment a
LEFT JOIN silver_sone_srmapping m
    ON CAST(a.appointment_status AS STRING) = CAST(m.id AS STRING)
WHERE a.appointment_status IS NOT NULL
ORDER BY 1;

In [ ]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT a.appointment_status) AS unique_status_ids,
    COUNT(DISTINCT m.mapping) AS unique_status_names
FROM silver_sone_srappointment a
LEFT JOIN silver_sone_srmapping m
    ON CAST(a.appointment_status AS STRING) = CAST(m.id AS STRING)
WHERE a.appointment_status IS NOT NULL;

In [ ]:
%%sql
SELECT DISTINCT
    a.appointment_status
FROM silver_sone_srappointment a
LEFT JOIN silver_sone_srmapping m
    ON CAST(a.appointment_status AS STRING) = CAST(m.id AS STRING)
WHERE a.appointment_status IS NOT NULL
  AND m.id IS NULL
ORDER BY a.appointment_status;

In [ ]:
%%sql
SELECT COUNT(*) AS s1_rows_to_add
FROM (
    SELECT DISTINCT
        CAST(a.appointment_status AS STRING) AS session_status_src_id,
        TRIM(m.mapping) AS session_status_src_name,
        'SONE' AS session_status_src_sys_inst_id
    FROM silver_sone_srappointment a
    LEFT JOIN silver_sone_srmapping m
        ON CAST(a.appointment_status AS STRING) = CAST(m.id AS STRING)
    WHERE a.appointment_status IS NOT NULL
      AND m.id IS NOT NULL
      AND m.mapping IS NOT NULL
      AND TRIM(m.mapping) <> ''
) s
LEFT JOIN silver_rdm_session_status r
    ON LOWER(TRIM(s.session_status_src_id)) = LOWER(TRIM(r.session_status_src_id))
WHERE r.session_status_src_id IS NULL;


In [ ]:
,
s1_source AS (
    -- S1 source values derived from SRAppointment and SRMapping
    -- appointment_status provides the source id and SRMapping.mapping provides the source name for SONE

    SELECT DISTINCT
        CAST(a.appointment_status AS STRING) AS session_status_src_id,
        TRIM(m.mapping) AS session_status_src_name,
        'SONE' AS session_status_src_sys_inst_id
    FROM silver_sone_srappointment a
    LEFT JOIN silver_sone_srmapping m
        ON CAST(a.appointment_status AS STRING) = CAST(m.id AS STRING)
    WHERE a.appointment_status IS NOT NULL
      AND m.id IS NOT NULL
      AND m.mapping IS NOT NULL
      AND TRIM(m.mapping) <> ''
)

In [ ]:
FROM (
    SELECT * FROM mpb_source
    UNION ALL
    SELECT * FROM wip_source
    UNION ALL
    SELECT * FROM s1_source
) s

In [ ]:
Implemented S1 session status logic in silver_rdm_session_status_add.

Created attributes:
- session_status_src_id
- session_status_src_name
- session_status_src_sys_inst_id

Source tables reviewed:
- silver_sone_srappointment
- silver_sone_srmapping

Join used:
- silver_sone_srappointment.appointment_status = silver_sone_srmapping.id

Implemented logic:
- session_status_src_id derived from silver_sone_srappointment.appointment_status
- session_status_src_name derived from silver_sone_srmapping.mapping
- session_status_src_sys_inst_id = SONE

Validation:
- total S1 rows to add: 17
- unique status ids: 17
- unique status names: 17
- unmatched mapping check returned no rows
- duplicate check returned no rows

Note:
S1 session status mapping is fully supported by the currently reviewed Silver source tables.